# v603 repo-owned verification

Copied from public Anatoly Iter5 SED v24 source for private repo-owned runtime/output preflight. No competition submit in this kernel run.


In [ ]:
import os
print('=== /kaggle/input/ ===')
if os.path.exists('/kaggle/input'):
    print(' ', os.listdir('/kaggle/input'))
os.makedirs('/kaggle/working/src', exist_ok=True)
open('/kaggle/working/src/model.py','w').write("\"\"\"SED model \u2014 Babich BC2025 1st-place architecture adapted for BC2026.\n\nCNN backbone (efficientnet/regnety/eca_nfnet via timm) + SED head with:\n- GeM frequency pooling (learnable p)\n- Attention pooling over time\n- Outputs: clipwise logits (B, C) + framewise logits (B, C, T)\n\nReference: https://www.kaggle.com/competitions/birdclef-2025/writeups/nikita-babych-1st-place\n\"\"\"\nfrom __future__ import annotations\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport timm\n\n\nclass GeMPool(nn.Module):\n    \"\"\"Generalized Mean pooling with learnable p.\"\"\"\n    def __init__(self, p: float = 3.0, eps: float = 1e-6):\n        super().__init__()\n        self.p = nn.Parameter(torch.tensor([p], dtype=torch.float32))\n        self.eps = eps\n\n    def forward(self, x: torch.Tensor, dim: int = -1) -> torch.Tensor:\n        x = x.clamp(min=self.eps).pow(self.p)\n        return x.mean(dim=dim).pow(1.0 / self.p)\n\n\nclass AttentionBlock(nn.Module):\n    \"\"\"Attention pooling over time for SED head.\n\n    From Kong et al. PANNs / Babich's adaptation.\n    \"\"\"\n    def __init__(self, in_features: int, out_features: int):\n        super().__init__()\n        self.att = nn.Linear(in_features, out_features)\n        self.cla = nn.Linear(in_features, out_features)\n        self.bn = nn.BatchNorm1d(out_features)\n\n    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:\n        \"\"\"x: (B, T, F) \u2192 clipwise_logits (B, C), framewise_logits (B, C, T).\n\n        Returns raw logits \u2014 apply sigmoid in inference, use BCEWithLogits in train.\n        \"\"\"\n        norm_att = torch.softmax(self.att(x).transpose(-1, -2).clamp(-10, 10), dim=-1)\n        cla_logit = self.cla(x).transpose(-1, -2)  # (B, C, T) raw logits\n        # Attention-weighted aggregation: clip_logit = sum_t (att_t \u00d7 cla_logit_t)\n        clip_logit = (norm_att * cla_logit).sum(dim=-1)\n        return clip_logit, cla_logit\n\n\nclass SEDModel(nn.Module):\n    \"\"\"Backbone + SED head (Babich-style).\"\"\"\n\n    def __init__(\n        self,\n        backbone_name: str = \"tf_efficientnet_b0.ns_jft_in1k\",\n        n_classes: int = 234,\n        drop_path_rate: float = 0.15,\n        gem_p: float = 3.0,\n        pretrained: bool = True,\n    ):\n        super().__init__()\n        self.backbone = timm.create_model(\n            backbone_name,\n            pretrained=pretrained,\n            features_only=True,\n            drop_path_rate=drop_path_rate,\n        )\n        feat_dim = self.backbone.feature_info.channels()[-1]\n\n        self.freq_pool = GeMPool(p=gem_p)\n        self.fc1 = nn.Linear(feat_dim, feat_dim, bias=True)\n        self.att_block = AttentionBlock(feat_dim, n_classes)\n\n    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:\n        \"\"\"x: (B, 3, F, T) mel-spectrogram (repeated 3 channels).\n\n        Returns:\n          {'clipwise_prob': (B, C), 'segmentwise_logit': (B, C, T)}\n        \"\"\"\n        feats = self.backbone(x)[-1]  # (B, feat_dim, f', t')\n        x = self.freq_pool(feats, dim=-2)  # GeM over frequency: (B, feat_dim, t')\n        x = x.transpose(1, 2)              # (B, t', feat_dim)\n        x = F.relu_(self.fc1(x))\n        clip_logit, frame_logit = self.att_block(x)\n        return {\n            \"clipwise_logit\": clip_logit,\n            \"segmentwise_logit\": frame_logit,\n            \"clipwise_prob\": torch.sigmoid(clip_logit),  # convenience for inference\n        }\n\n\ndef sed_loss(outputs: dict, targets: torch.Tensor) -> dict[str, torch.Tensor]:\n    \"\"\"Babich's combined SED loss: 0.5 BCE(clip_logit) + 0.5 BCE(frame_logit.max).\n\n    Uses BCE_with_logits (AMP-safe). Inputs are raw logits.\n    \"\"\"\n    clip_logit = outputs[\"clipwise_logit\"]\n    frame_max_logit = outputs[\"segmentwise_logit\"].max(dim=2)[0]\n    loss_clip = F.binary_cross_entropy_with_logits(clip_logit, targets)\n    loss_frame = F.binary_cross_entropy_with_logits(frame_max_logit, targets)\n    return {\n        \"loss\": 0.5 * loss_clip + 0.5 * loss_frame,\n        \"loss_clip\": loss_clip.detach(),\n        \"loss_frame\": loss_frame.detach(),\n    }\n")
open('/kaggle/working/src/dataset.py','w').write("\"\"\"SED dataset \u2014 20-sec audio chunks + MixUp + mel-spec processor.\n\nBabich mel params: n_mels=224, hop=1252, n_fft=4096, top_db=80, fmin=0, fmax=16000.\nSample rate: 32000. Chunk: 20 sec = 640000 samples.\n\nMixUp on raw waveforms (NOT spectrograms) with constant blend=0.5.\n\"\"\"\nfrom __future__ import annotations\nimport random\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport torchaudio\nfrom torch.utils.data import Dataset\nfrom audio_loader import load_audio\n\n\nCHUNK_SEC = 20\nSAMPLE_RATE = 32000\nCHUNK_SAMPLES = CHUNK_SEC * SAMPLE_RATE  # 640000\n\n# Mel cache: HDF5 with full-file mel-specs keyed by file stem (Tier 2 enabler, 5\u00d7 speedup).\n# Set via dataset.set_mel_cache(path) ONCE before DataLoader creation.\n_MEL_CACHE_PATH: str | None = None\n_MEL_CACHE_HANDLE = None  # lazy h5py.File per worker\n\n\ndef set_mel_cache(path: str | None) -> None:\n    \"\"\"Configure global mel cache HDF5 path. Set ONCE before DataLoader init.\"\"\"\n    global _MEL_CACHE_PATH, _MEL_CACHE_HANDLE\n    _MEL_CACHE_PATH = path\n    _MEL_CACHE_HANDLE = None  # force re-open in workers\n\n\ndef _get_mel_handle():\n    \"\"\"Lazy-open h5py handle per worker (multi-proc safe).\"\"\"\n    global _MEL_CACHE_HANDLE\n    if _MEL_CACHE_PATH is None:\n        return None\n    if _MEL_CACHE_HANDLE is None:\n        import h5py\n        _MEL_CACHE_HANDLE = h5py.File(_MEL_CACHE_PATH, \"r\", swmr=True)\n    return _MEL_CACHE_HANDLE\n\n\ndef _hop_frames_per_sec(hop_length: int = 1252, sr: int = SAMPLE_RATE) -> float:\n    \"\"\"Mel frames per second of audio.\"\"\"\n    return sr / hop_length  # ~25.56 fps for our params\n\n\nCHUNK_MEL_FRAMES = int(CHUNK_SEC * _hop_frames_per_sec()) + 1  # ~512 frames per 20s chunk\n\n\ndef load_mel_chunk_from_cache(file_stem: str, start_sec: float, sample_rate: int = SAMPLE_RATE) -> np.ndarray | None:\n    \"\"\"Read mel-spec slice for given chunk from cache. Returns None if not cached.\n\n    Returns mel of shape (n_mels, CHUNK_MEL_FRAMES) or None.\n    \"\"\"\n    h5 = _get_mel_handle()\n    if h5 is None:\n        return None\n    if file_stem not in h5:\n        return None\n    full_mel = h5[file_stem]  # (n_mels, n_frames)\n    fps = _hop_frames_per_sec()\n    s = max(0, int(start_sec * fps))\n    e = s + CHUNK_MEL_FRAMES\n    n_frames = full_mel.shape[1]\n    if e > n_frames:\n        # pad with -80 dB (silence in db scale)\n        out = np.full((full_mel.shape[0], CHUNK_MEL_FRAMES), -80.0, dtype=np.float32)\n        avail = max(0, n_frames - s)\n        if avail > 0:\n            out[:, :avail] = full_mel[:, s:n_frames].astype(np.float32)\n        return out\n    return full_mel[:, s:e].astype(np.float32)\n\nMEL_KWARGS = dict(\n    sample_rate=SAMPLE_RATE,\n    n_fft=4096,\n    hop_length=1252,\n    n_mels=224,\n    f_min=0,\n    f_max=16000,\n    power=2.0,\n)\n\n\ndef pad_wave(wave: np.ndarray, expected_len: int = CHUNK_SAMPLES, pad_type: str = \"random\") -> np.ndarray:\n    \"\"\"Pad short waves to expected length. Babich's helper.\n\n    pad_type: 'random' inserts at random position, 'left' pads at start, 'repeat' tiles.\n    \"\"\"\n    if wave.shape[0] >= expected_len:\n        return wave[:expected_len]\n    pad_len = expected_len - wave.shape[0]\n    if pad_type == \"random\":\n        out = np.zeros(expected_len, dtype=wave.dtype)\n        start = np.random.randint(0, pad_len + 1)\n        out[start : start + wave.shape[0]] = wave\n        return out\n    if pad_type == \"left\":\n        return np.pad(wave, ((pad_len, 0)))\n    if pad_type == \"repeat\":\n        reps = int(np.ceil(expected_len / wave.shape[0]))\n        return np.tile(wave, reps)[:expected_len]\n    raise ValueError(pad_type)\n\n\ndef absmax_normalize(wave: np.ndarray) -> np.ndarray:\n    \"\"\"Normalize wave by abs-max.\"\"\"\n    m = np.abs(wave).max()\n    if m < 1e-8:\n        return wave\n    return wave / m\n\n\nclass MelCachedDataset(Dataset):\n    \"\"\"Mel cache-aware dataset wrapper. Reads mel from HDF5 cache \u0432 /dev/shm.\n\n    Falls back to audio decode if cache miss (for backward compat).\n    Returns (mel_tensor, target) \u2014 same interface as AudioDataset but mel pre-computed.\n\n    Drop-in replacement for AudioDataset / PseudoAudioDataset when --mel-cache flag set.\n    Skips audio decode entirely if file in cache \u2192 eliminates CPU bottleneck.\n\n    Mel shape: (n_mels=224, CHUNK_MEL_FRAMES=~512) \u2014 fixed-size chunk for 20s audio.\n    \"\"\"\n\n    def __init__(\n        self,\n        manifest: list[dict] | pd.DataFrame,\n        class_to_idx: dict[str, int] | None = None,\n        classes: list[str] | None = None,\n        mode: str = \"focal\",  # \"focal\" (manifest with labels) or \"pseudo\" (df with soft probs)\n        augment: bool = True,\n    ):\n        # Defer import to avoid circular\n        global pd\n        try:\n            import pandas as pd  # noqa\n        except Exception:\n            pass\n        self.mode = mode\n        self.augment = augment\n        if isinstance(manifest, list):\n            self.records = manifest\n            self.is_df = False\n        else:\n            self.records = manifest\n            self.is_df = True\n        self.class_to_idx = class_to_idx\n        self.classes = classes\n        self.n_classes = len(class_to_idx) if class_to_idx else (len(classes) if classes else 234)\n        # For pseudo mode: cache column names of class probs\n        if mode == \"pseudo\" and self.is_df:\n            self.class_cols = [c for c in classes if c in self.records.columns] if classes else []\n\n    def __len__(self) -> int:\n        return len(self.records)\n\n    def __getitem__(self, idx: int):\n        rec = self.records.iloc[idx] if self.is_df else self.records[idx]\n        # Get filename + start_sec (mode-dependent extraction)\n        if self.mode == \"focal\":\n            path = rec.get(\"path\", rec.get(\"filepath\", rec.get(\"filename\", \"\")))\n            start_sec = float(rec.get(\"start_sec\", 0.0))\n        else:  # pseudo\n            path = rec[\"filename\"]\n            start_sec = float(rec[\"start_sec\"])\n        file_stem = Path(str(path)).stem\n        # Augmentation: random start jitter for focal training\n        if self.augment and self.mode == \"focal\":\n            # Random small offset within \u00b12s (mel frames)\n            jitter_sec = (random.random() - 0.5) * 4.0  # -2 to +2 sec\n            start_sec = max(0.0, start_sec + jitter_sec)\n        # Load mel from cache\n        mel = load_mel_chunk_from_cache(file_stem, start_sec)\n        if mel is None:\n            # Cache miss \u2192 fallback to audio decode + mel compute\n            wave = self._load_audio_fallback(rec)\n            # Compute mel on-the-fly (slower)\n            wave_t = torch.from_numpy(wave).float().unsqueeze(0)\n            mel_tf = torchaudio.transforms.MelSpectrogram(**MEL_KWARGS)\n            db_tf = torchaudio.transforms.AmplitudeToDB(top_db=80.0)\n            mel = mel_tf(wave_t).numpy().squeeze(0)\n            mel = db_tf(torch.from_numpy(mel)).numpy()\n        # Normalize 0-1 per chunk\n        m_min, m_max = mel.min(), mel.max()\n        if m_max - m_min > 1e-8:\n            mel = (mel - m_min) / (m_max - m_min)\n        # Build target\n        target = self._make_target(rec)\n        return torch.from_numpy(mel).float(), torch.from_numpy(target).float()\n\n    def _load_audio_fallback(self, rec) -> np.ndarray:\n        \"\"\"Fallback audio decode for cache miss.\"\"\"\n        path = rec.get(\"path\", rec.get(\"filepath\", rec.get(\"filename\", \"\"))) if not self.is_df else (\n            rec[\"filepath\"] if \"filepath\" in rec.index else rec[\"filename\"]\n        )\n        wave, _ = load_audio(str(path), target_sr=SAMPLE_RATE)\n        wave = wave.squeeze(0).numpy()\n        start_sec = float(rec.get(\"start_sec\", 0.0))\n        end_sec = float(rec.get(\"end_sec\", start_sec + CHUNK_SEC))\n        s = int(start_sec * SAMPLE_RATE)\n        e = int(end_sec * SAMPLE_RATE)\n        chunk = wave[s:e] if e > s else wave[:CHUNK_SAMPLES]\n        chunk = absmax_normalize(chunk)\n        chunk = pad_wave(chunk, CHUNK_SAMPLES, pad_type=\"random\" if self.augment else \"left\")\n        return chunk\n\n    def _make_target(self, rec) -> np.ndarray:\n        target = np.zeros(self.n_classes, dtype=np.float32)\n        if self.mode == \"pseudo\":\n            # Soft probs from pseudo manifest\n            for ci, c in enumerate(self.classes):\n                target[ci] = float(rec[c]) if c in (rec.index if self.is_df else rec) else 0.0\n        else:  # focal: hard labels via labels column\n            labels = rec.get(\"labels\", rec.get(\"primary_label\", \"\"))\n            if isinstance(labels, str):\n                labels = labels.split(\";\")\n            elif not hasattr(labels, \"__iter__\"):\n                labels = [labels]\n            for lab in labels:\n                lab = str(lab).strip()\n                if lab and lab in (self.class_to_idx or {}):\n                    target[self.class_to_idx[lab]] = 1.0\n        return target\n\n\nclass AudioDataset(Dataset):\n    \"\"\"Focal audio dataset for SED Stage 1.\n\n    Each item: 20-sec chunk + multi-label target (n_classes,).\n    Optional MixUp applied at batch level (handled externally in collate).\n    \"\"\"\n\n    def __init__(\n        self,\n        manifest: list[dict],\n        class_to_idx: dict[str, int],\n        sample_rate: int = SAMPLE_RATE,\n        chunk_samples: int = CHUNK_SAMPLES,\n        augment: bool = True,\n    ):\n        self.manifest = manifest\n        self.class_to_idx = class_to_idx\n        self.n_classes = len(class_to_idx)\n        self.sample_rate = sample_rate\n        self.chunk_samples = chunk_samples\n        self.augment = augment\n\n    def __len__(self) -> int:\n        return len(self.manifest)\n\n    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:\n        rec = self.manifest[idx]\n        wave = self._load_audio(rec[\"path\"])\n\n        # Window extraction for ARU rows (start_sec/end_sec \u2265 0)\n        start_sec = rec.get(\"start_sec\", -1)\n        end_sec = rec.get(\"end_sec\", -1)\n        if start_sec >= 0 and end_sec > start_sec:\n            s, e = int(start_sec * self.sample_rate), int(end_sec * self.sample_rate)\n            wave = wave[s:e]\n\n        wave = absmax_normalize(wave)\n        wave = pad_wave(wave, self.chunk_samples, pad_type=\"random\" if self.augment else \"left\")\n\n        target = self._make_target(rec)\n        return torch.from_numpy(wave).float(), torch.from_numpy(target).float()\n\n    def _load_audio(self, path: str) -> np.ndarray:\n        wave, sr = load_audio(path, target_sr=self.sample_rate)\n        return wave.squeeze(0).numpy()  # [1, N] \u2192 [N]\n\n    def _make_target(self, rec: dict) -> np.ndarray:\n        t = np.zeros(self.n_classes, dtype=np.float32)\n        # Multi-label support: \"labels\" field may be \";\"-separated string OR list\n        labels = rec.get(\"labels\")\n        if labels is None:\n            labels = [rec.get(\"primary_label\", \"\")]\n        if isinstance(labels, str):\n            labels = labels.split(\";\")\n        for lab in labels:\n            lab = str(lab).strip()\n            if lab and lab in self.class_to_idx:\n                t[self.class_to_idx[lab]] = 1.0\n        return t\n\n\ndef mixup_collate(batch, blend: float = 0.5) -> tuple[torch.Tensor, torch.Tensor]:\n    \"\"\"Babich's MixUp on raw waves: constant blend=0.5, max() of targets.\n\n    Returns: (waves, targets) where each item is a mix of two random pairs.\n    \"\"\"\n    waves, targets = zip(*batch)\n    waves = torch.stack(waves)\n    targets = torch.stack(targets)\n    perm = torch.randperm(waves.size(0))\n    waves_mixed = blend * waves + (1.0 - blend) * waves[perm]\n    targets_mixed = torch.max(targets, targets[perm])  # union of labels\n    return waves_mixed, targets_mixed\n\n\nclass MelTransform:\n    \"\"\"Compute mel-spec on the fly. Repeated 3 channels for ImageNet-pretrained CNN.\n\n    Optional SpecAugment (training-time only) via apply call with training=True:\n      - freq_mask: random N consecutive mel-bins masked\n      - time_mask: random N consecutive time frames masked\n    Babich BC2025 used spec-augment with freq_mask=24, time_mask=80, n_masks=2.\n    \"\"\"\n\n    def __init__(self, freq_mask_param: int = 24, time_mask_param: int = 80, n_masks: int = 2, **kwargs):\n        cfg = {**MEL_KWARGS, **kwargs}\n        self.mel = torchaudio.transforms.MelSpectrogram(**cfg)\n        self.db = torchaudio.transforms.AmplitudeToDB(top_db=80.0)\n        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param) if freq_mask_param > 0 else None\n        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param) if time_mask_param > 0 else None\n        self.n_masks = n_masks\n\n    def __call__(self, wave: torch.Tensor, training: bool = False) -> torch.Tensor:\n        \"\"\"wave: (B, T) \u2192 (B, 3, n_mels, n_frames). If training=True, apply SpecAugment.\"\"\"\n        mel = self.mel(wave)  # (B, n_mels, n_frames)\n        mel = self.db(mel)\n        # 0-1 normalize per sample\n        mel_min = mel.amin(dim=(-1, -2), keepdim=True)\n        mel_max = mel.amax(dim=(-1, -2), keepdim=True)\n        mel = (mel - mel_min) / (mel_max - mel_min + 1e-8)\n        # SpecAugment \u2014 applied to 2D mel BEFORE channel repeat\n        if training and self.freq_mask is not None:\n            for _ in range(self.n_masks):\n                mel = self.freq_mask(mel)\n        if training and self.time_mask is not None:\n            for _ in range(self.n_masks):\n                mel = self.time_mask(mel)\n        mel = mel.unsqueeze(1).repeat(1, 3, 1, 1)  # (B, 3, n_mels, n_frames)\n        return mel\n")
open('/kaggle/working/src/audio_loader.py','w').write("\"\"\"Audio loader replacing torchaudio.load \u2014 uses soundfile (works with PyTorch 2.11+).\n\ntorchaudio 2.7+ deprecates the native load() in favor of torchcodec, which has\nABI issues with stable PyTorch 2.11/cu128. soundfile is a robust pure-Python\nbackend over libsndfile, no torch coupling.\n\nReturns (wave_tensor_1xN, sr) matching old torchaudio.load API.\n\"\"\"\nfrom __future__ import annotations\nimport numpy as np\nimport soundfile as sf\nimport torch\nimport torchaudio.functional as F\n\n\ndef load_audio(path: str, target_sr: int | None = None) -> tuple[torch.Tensor, int]:\n    \"\"\"Drop-in replacement for torchaudio.load() + optional resample + mono.\n\n    Args:\n        path: path to audio file (.ogg/.wav/.flac/.mp3)\n        target_sr: if set and differs from file's sr, resample.\n\n    Returns:\n        (wave [1, N] float32 tensor, sr int)\n    \"\"\"\n    wave, sr = sf.read(path, dtype=\"float32\", always_2d=False)\n    # soundfile returns (N,) for mono or (N, C) for multi-channel\n    if wave.ndim == 2:\n        wave = wave.mean(axis=1)  # to mono\n    wave_t = torch.from_numpy(wave).unsqueeze(0)  # [1, N]\n    if target_sr is not None and target_sr != sr:\n        wave_t = F.resample(wave_t, sr, target_sr)\n        sr = target_sr\n    return wave_t, sr\n")
open('/kaggle/working/src/kernel.py','w').write('"""BC2026 — 5-srx ensemble submit (built on Iter 5 single-model kernel)."""\nfrom __future__ import annotations\nimport argparse, json, sys, traceback, time, re\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\n\nKAGGLE_WORKING = Path("/kaggle/working")\nTARGET_BACKBONE = "tf_efficientnet_b0.ns_jft_in1k"\nTARGET_FOLDS = [1]  # b0_fold1 val 0.9579, ~3x faster on CPU than srx\n\n\ndef parse_args():\n    p = argparse.ArgumentParser()\n    p.add_argument("--ckpt-dir", required=True)\n    p.add_argument("--test-soundscape-dir", required=True)\n    p.add_argument("--sample-submission", required=True)\n    p.add_argument("--classes", default="")\n    p.add_argument("--batch", type=int, default=32)\n    p.add_argument("--out", default=str(KAGGLE_WORKING / "submission.csv"))\n    p.add_argument("--limit", type=int, default=0)\n    return p.parse_args()\n\n\ndef find_ckpts(ckpt_dir: str) -> list[Path]:\n    """Locate b0 fold ckpts in dataset."""\n    found = []\n    for f in TARGET_FOLDS:\n        # Try standard layout\n        for root in ["iter5", "ckpts_iter5", "."]:\n            p = Path(ckpt_dir) / root / f"b0_fold{f}" / "best.pth"\n            if p.exists():\n                found.append(p); break\n        else:\n            # Fallback rglob\n            cand = sorted(Path(ckpt_dir).rglob(f"b0_fold{f}/best.pth"))\n            if cand:\n                found.append(cand[0])\n    return found\n\n\ndef main():\n    args = parse_args()\n    fp16_use = False\n    use_cuda = False\n    if torch.cuda.is_available():\n        cap = torch.cuda.get_device_capability(0)\n        gpu_name = torch.cuda.get_device_name(0)\n        print(f"[kernel] GPU: {gpu_name} (sm_{cap[0]}{cap[1]})")\n        if cap[0] >= 7:\n            use_cuda = True\n            fp16_use = True\n        else:\n            print(f"[kernel] GPU cap < sm_70 — falling back to CPU (PyTorch sm_60 incompat)")\n    else:\n        print("[kernel] no CUDA — using CPU")\n\n    sys.path.insert(0, args.ckpt_dir)\n    from model import SEDModel\n    from dataset import MelTransform, SAMPLE_RATE, CHUNK_SAMPLES, absmax_normalize, pad_wave\n    from audio_loader import load_audio\n\n    sub_df = pd.read_csv(args.sample_submission)\n    sub_classes = [c for c in sub_df.columns if c != "row_id"]\n    print(f"[kernel] sample_sub: {len(sub_df)} rows × {len(sub_classes)} class cols")\n    print(f"[kernel] first row_id: {sub_df[\'row_id\'].iloc[0]}  last: {sub_df[\'row_id\'].iloc[-1]}")\n\n    rid_re = re.compile(r\'^(.+)_(\\d+)$\')\n    parsed_fnames, parsed_ends = [], []\n    for rid in sub_df[\'row_id\'].values:\n        m = rid_re.match(rid)\n        if m:\n            parsed_fnames.append(m.group(1)); parsed_ends.append(int(m.group(2)))\n        else:\n            parsed_fnames.append(rid); parsed_ends.append(0)\n    parsed_fnames = np.array(parsed_fnames); parsed_ends = np.array(parsed_ends)\n\n    fname_keys = list(dict.fromkeys(parsed_fnames.tolist()))\n    print(f"[kernel] {len(fname_keys)} unique filename keys; sample: {fname_keys[:3]}")\n    end_times_per = {}\n    for k in fname_keys:\n        end_times_per[k] = parsed_ends[parsed_fnames == k].tolist()\n    sample_endts = end_times_per[fname_keys[0]]\n    win_stride = sample_endts[1] - sample_endts[0] if len(sample_endts) >= 2 else 5\n    print(f"[kernel] n_win/file={len(sample_endts)} stride={win_stride}s endts[:3]={sample_endts[:3]}")\n\n    out_df = sub_df[[\'row_id\']].copy().reset_index(drop=True)\n    for c in sub_classes:\n        out_df[c] = 0.0\n\n    # SAFETY: write zeros submission upfront so any timeout still produces valid format\n    out_df.to_csv(args.out, index=False, float_format=\'%.6f\')\n    print(f"[kernel] safety submission written (all-zeros baseline): {len(out_df)} rows")\n\n    # Load ensemble\n    ckpt_paths = find_ckpts(args.ckpt_dir)\n    if not ckpt_paths:\n        raise RuntimeError("no seresnext ckpts found in dataset")\n    print(f"[kernel] found {len(ckpt_paths)} srx ckpts:")\n    for p in ckpt_paths:\n        print(f"  - {p}")\n\n    models = []\n    for ckp in ckpt_paths:\n        ck = torch.load(ckp, map_location="cpu", weights_only=False)\n        sd = ck.get("model", ck) if isinstance(ck, dict) else ck\n        m = SEDModel(backbone_name=TARGET_BACKBONE, n_classes=234, pretrained=False)\n        m.load_state_dict(sd, strict=False)\n        if fp16_use:\n            m = m.half()\n        m.eval()\n        if use_cuda:\n            m = m.cuda()\n        models.append(m)\n    print(f"[kernel] loaded {len(models)} models ({TARGET_BACKBONE}), fp16={fp16_use}")\n\n    mel_tf = MelTransform()\n    if use_cuda:\n        mel_tf.mel = mel_tf.mel.cuda()\n        mel_tf.db = mel_tf.db.cuda()\n\n    test_dir = Path(args.test_soundscape_dir)\n    test_files = sorted(test_dir.glob("*.ogg"))\n    if args.limit > 0:\n        test_files = test_files[:args.limit]\n    print(f"[kernel] {len(test_files)} test soundscapes")\n\n    if len(test_files) == 0:\n        print(f"[kernel] WARN: 0 .ogg in {test_dir}")\n        try:\n            import os as _os\n            for entry in sorted(_os.listdir(str(test_dir)))[:30]:\n                print(f"  listing: {entry}")\n        except Exception as _e:\n            print(f"  [listdir err: {_e}]")\n        # also list parent dir\n        try:\n            for entry in sorted(_os.listdir(str(test_dir.parent)))[:20]:\n                print(f"  parent: {entry}")\n        except Exception as _e:\n            print(f"  [parent err: {_e}]")\n        print("[kernel] hidden-test mode — zeros submission")\n        out_df.to_csv(args.out, index=False, float_format=\'%.6f\')\n        print(f"[kernel] stub rows={len(out_df)} cols={len(out_df.columns)}")\n        return\n\n    rid_to_idx = {r: i for i, r in enumerate(out_df[\'row_id\'].values)}\n    audio_target_len = sample_endts[-1] * SAMPLE_RATE\n    t_start = time.time()\n    matched = 0\n\n    for f_idx, fpath in enumerate(test_files):\n        try:\n            stem = fpath.stem\n            name = fpath.name\n            match_key = None\n            for k in (stem, name, name.split(\'.\')[0]):\n                if k in end_times_per:\n                    match_key = k; break\n            if match_key is None:\n                for k in end_times_per:\n                    if k.endswith(stem) or stem.endswith(k):\n                        match_key = k; break\n            if match_key is None:\n                print(f"[kernel] WARN no match for {fpath.name}, skip")\n                continue\n\n            wave, _ = load_audio(str(fpath), target_sr=SAMPLE_RATE)\n            wave = wave.squeeze(0).numpy()\n            if len(wave) < audio_target_len:\n                wave = np.pad(wave, (0, audio_target_len - len(wave)))\n            else:\n                wave = wave[:audio_target_len]\n\n            file_endts = end_times_per[match_key]\n            n_win = len(file_endts)\n            chunks = []\n            for et in file_endts:\n                start = (et - win_stride) * SAMPLE_RATE\n                end_s = et * SAMPLE_RATE\n                c = wave[start:end_s]\n                c = absmax_normalize(c)\n                c = pad_wave(c, CHUNK_SAMPLES, pad_type="left")\n                chunks.append(c)\n            chunks_np = np.stack(chunks, axis=0).astype(np.float32)\n\n            preds = np.zeros((n_win, 234), dtype=np.float32)\n            BS = args.batch\n            for s in range(0, n_win, BS):\n                e = min(s + BS, n_win)\n                wt = torch.from_numpy(chunks_np[s:e]).float()\n                if use_cuda:\n                    wt = wt.cuda()\n                mel = mel_tf(wt)\n                if fp16_use:\n                    mel = mel.half()\n                # Ensemble: mean of sigmoid probs across 5 models\n                ens_probs = np.zeros((e - s, 234), dtype=np.float32)\n                with torch.no_grad():\n                    for m in models:\n                        out = m(mel)\n                        p = torch.sigmoid(out["clipwise_logit"]).float().cpu().numpy()\n                        ens_probs += p\n                ens_probs /= len(models)\n                ens_probs = np.nan_to_num(ens_probs, nan=0.0, posinf=1.0, neginf=0.0)\n                ens_probs = np.clip(ens_probs, 0.0, 1.0)\n                preds[s:e] = ens_probs\n\n            n_cls = min(234, len(sub_classes))\n            for wi, et in enumerate(file_endts):\n                rid = f"{match_key}_{et}"\n                if rid in rid_to_idx:\n                    ridx = rid_to_idx[rid]\n                    out_df.iloc[ridx, 1:n_cls + 1] = preds[wi, :n_cls]\n            matched += 1\n\n        except Exception as e:\n            print(f"[kernel] FAILED {fpath.name}: {type(e).__name__}: {e}")\n            traceback.print_exc()\n\n        if f_idx % 100 == 0 and f_idx > 0:\n            # Progressive save — safety against timeout\n            out_df.to_csv(args.out, index=False, float_format=\'%.6f\')\n            print(f"[kernel] progressive save at {f_idx}/{len(test_files)}")\n            if use_cuda:\n                torch.cuda.empty_cache()\n            el = time.time() - t_start\n            rate = (f_idx + 1) / el if el > 0 else 0\n            eta = (len(test_files) - f_idx - 1) / rate / 60 if rate > 0 else 0\n            print(f"[kernel] {f_idx + 1}/{len(test_files)} rate={rate:.2f}/s eta={eta:.1f}min")\n\n    assert len(out_df) == len(sub_df), f"rows {len(out_df)} != {len(sub_df)}"\n    assert list(out_df.columns) == list(sub_df.columns), "col mismatch"\n    for c in sub_classes:\n        out_df[c] = pd.to_numeric(out_df[c], errors=\'coerce\').fillna(0.0).clip(0.0, 1.0).astype(\'float32\')\n\n    out_df.to_csv(args.out, index=False, float_format=\'%.6f\')\n    print(f"[kernel] saved rows={len(out_df)} cols={len(out_df.columns)} matched={matched} elapsed={(time.time()-t_start)/60:.1f}min")\n\n\nif __name__ == "__main__":\n    main()\n')
print('sources written')


In [ ]:
import subprocess, sys, os
ds_root = None
comp_root = None
for r, dirs, files in os.walk('/kaggle/input'):
    if 'iter5_logreg.npz' in files and ds_root is None:
        ds_root = r
    if 'sample_submission.csv' in files and 'test_soundscapes' in dirs and comp_root is None:
        comp_root = r
    if ds_root and comp_root: break
if ds_root is None: raise RuntimeError('ds_root not found')
if comp_root is None: raise RuntimeError('comp_root not found')
print('ds_root:', ds_root); print('comp_root:', comp_root)
env = os.environ.copy()
env['PYTHONPATH'] = '/kaggle/working/src:' + env.get('PYTHONPATH', '')
cmd = [sys.executable, '/kaggle/working/src/kernel.py',
       '--ckpt-dir', ds_root,
       '--test-soundscape-dir', os.path.join(comp_root, 'test_soundscapes'),
       '--sample-submission', os.path.join(comp_root, 'sample_submission.csv'),
       '--classes', os.path.join(comp_root, 'taxonomy.csv')]
print('running:', ' '.join(cmd))
r = subprocess.run(cmd, env=env)
if r.returncode != 0: raise RuntimeError(f'kernel exit {r.returncode}')
print('kernel done OK')
